# Colab 01 — Búsqueda de hiperparámetros (Entregable 3, Bloque B)

Entrena **8 intentos por modelo** (intento 0 = configuración del protocolo, 7 aleatorios; **1 semilla (42)** por intento) para 10 modelos: `unet_fuerte`, `attention_unet` (U-Net + scSE), `unet_pp`, `deeplabv3p`, `transunet`, `swin_unet`, `unetr`, `segformer`, `mednext` y el modelo original `lacca_unet`. `unet` se entrena aparte, sin búsqueda; nnU-Net tiene su propio pipeline.

**Garantías**
- Solo lee `train` y `val`. El test sellado NO se copia ni se toca.
- Selección únicamente por `dice_val`.
- Reanudable: si la sesión se corta, **vuelve a ejecutar la celda 8** hasta que diga `pendientes 0`.
- Borra los pesos de cada intento (solo importan los resultados) para no llenar Drive.

**Tiempo estimado:** 80 intentos; con ~10–15 min por intento en una T4/L4 son ~15 h, es decir, unas 3 sesiones de 5,5 h. Runtime → Change runtime type → GPU.

**Antes de empezar:** en Drive deben estar `proyecto_tb/codigo/` (con `original_model.py`, `hpo_busqueda.py`, `tests/`) y `proyecto_tb/datos/splits/`.

In [ ]:
# Celda 1 — GPU
import os
!nvidia-smi | head -15
print("\nSi no aparece una GPU: Runtime → Change runtime type → GPU (T4 o L4).")

In [ ]:
# Celda 2 — dependencias (sin nnunetv2: no se necesita para la búsqueda)
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm pytest
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
assert torch.cuda.is_available(), "Sin GPU: activa una GPU antes de seguir"

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')

CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"   # AJUSTAR si cambia
RAIZ_LOCAL = "/content/proyecto_tb"                     # copia local (rápida)
RESULTADOS = f"{CARPETA_DRIVE}/resultados"              # en Drive: sobrevive

assert os.path.isdir(f"{CARPETA_DRIVE}/codigo"), "Falta codigo/ en Drive"
assert os.path.isfile(f"{CARPETA_DRIVE}/codigo/hpo_busqueda.py"), "Falta hpo_busqueda.py"
assert os.path.isfile(f"{CARPETA_DRIVE}/codigo/original_model.py"), "Falta original_model.py"
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits"
print("Drive OK:", CARPETA_DRIVE)

In [ ]:
# Celda 4 — copiar código y SOLO train/val a disco local (el test no se copia)
import csv, shutil

def a_local(ruta_windows, raiz):
    norm = ruta_windows.replace("\\", "/")
    i = norm.find("datos/")
    assert i >= 0, f"ruta sin prefijo datos/: {ruta_windows}"
    return os.path.join(raiz, norm[i:])

shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo", dirs_exist_ok=True)
os.makedirs(f"{RAIZ_LOCAL}/datos/splits", exist_ok=True)
n_nuevos = 0
for nombre in ("train", "val"):
    with open(f"{CARPETA_DRIVE}/datos/splits/{nombre}.csv", newline="", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))
    campos = list(filas[0].keys())
    for fila in filas:
        dst = a_local(fila["ruta"], RAIZ_LOCAL)
        src = a_local(fila["ruta"], CARPETA_DRIVE)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            n_nuevos += 1
        fila["ruta"] = dst
    with open(f"{RAIZ_LOCAL}/datos/splits/{nombre}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos); w.writeheader(); w.writerows(filas)
    print(f"  split {nombre:6s}: {len(filas)} filas")
for mf in ("split_manifest_v2.json", "split_manifest.json"):
    if os.path.exists(f"{CARPETA_DRIVE}/datos/splits/{mf}"):
        shutil.copy2(f"{CARPETA_DRIVE}/datos/splits/{mf}", f"{RAIZ_LOCAL}/datos/splits/{mf}")
print("archivos de datos copiados:", n_nuevos, "| test NO copiado (por diseño)")

In [ ]:
# Celda 5 — control de calidad: pruebas unitarias del modelo original + entorno
!cd "{RAIZ_LOCAL}" && python -m pytest codigo/tests -q 2>&1 | tail -5
import sys
sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")
from modelos_tb import resumen_entorno
resumen_entorno()

In [ ]:
# Celda 6 — plan (no entrena): 80 intentos, con sus hiperparámetros
!python "{RAIZ_LOCAL}/codigo/hpo_busqueda.py" --dry-run --splits-dir "{RAIZ_LOCAL}/datos/splits" --resultados "{RESULTADOS}"

In [ ]:
# Celda 7 — prueba de humo (1 época, 1 intento, a una carpeta temporal; ~2 min)
# Comprueba de punta a punta que datos + modelo + pérdida + guardado funcionan.
!python "{RAIZ_LOCAL}/codigo/hpo_busqueda.py" --modelos unet_fuerte lacca_unet --intentos 1 --epochs 1 --splits-dir "{RAIZ_LOCAL}/datos/splits" --resultados /content/hpo_humo

## Búsqueda real
Ejecuta la celda 8. Corta limpio a los 330 min y guarda. **Repite la misma celda en la siguiente sesión** hasta que el plan diga `pendientes 0`.

In [ ]:
# Celda 8 — búsqueda de hiperparámetros (reanudable)
!python "{RAIZ_LOCAL}/codigo/hpo_busqueda.py" --modelos transunet swin_unet "{RAIZ_LOCAL}/datos/splits" --resultados "{RESULTADOS}" --max-minutos 330

In [ ]:
# Celda 9 — progreso y mejores configuraciones hasta ahora
import pandas as pd, json
p = f"{RESULTADOS}/hpo/hpo_intentos.csv"
if os.path.exists(p):
    df = pd.read_csv(p)
    print(f"{len(df)} intentos terminados de 80")
    print(df.groupby("modelo").agg(intentos=("intento","size"),
          mejor_dice=("dice_val","max"), dice_por_defecto=("dice_val", lambda s: s.iloc[0])).round(4))
q = f"{RESULTADOS}/hpo/mejor_config.json"
if os.path.exists(q):
    print(json.dumps({k: {x: v[x] for x in ("lr","weight_decay","batch_size","_intento")} for k, v in json.load(open(q)).items()}, indent=1))

## Al terminar
Cuando el plan diga `pendientes 0`, `resultados/hpo/mejor_config.json` queda en Drive. Ese archivo lo usa `colab_02_entrenamiento_final.ipynb` con `--config-hpo`. **Avísame para revisar los resultados antes del entrenamiento final** (en particular, si algún modelo no mejoró sobre su configuración por defecto).